# Merge related tables · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 17. Merge related tables

Connect tables by a key and audit which records matched.

Before you start: Use column selection and understand unique Student_ID values.

Official reference: https://pandas.pydata.org/docs/user_guide/merging.html

### M17C01. Attach the department

The Program column is a key shared by two tables. A many-to-one merge lets several students match one department row. The validation checks that the lookup table has only one row per program.

**Read:** What does on="Program" tell pandas?

Answer: Match the two tables using Program values.

**Predict:** Will the result contain 2 rows or 12 rows?

Answer: 12 rows, one for each student, because each Program matches exactly one lookup row.

**Explain:** Why validate="many_to_one"?

Answer: Students may repeat a Program, but each Program must appear at most once in the lookup table.



In [ ]:
# Predict before running: Will the result contain 2 rows or 12 rows?
df = students()
result = df.merge(departments(), on="Program", how="left", validate="many_to_one")
print(result[["Student_ID", "Program", "Department"]])

**Your turn:** Use DataFrame.join to attach the same lookup fields. Set Program as the lookup index and keep the student index.

For the check, keep these result names: `result`.

**Hint:** Use departments().set_index("Program"), then join(..., on="Program").

**Watch for this:** Do not join tables by row position when they describe entities identified by keys.

In [ ]:
# Change this code to complete the task.
# Question: Use DataFrame.join to attach the same lookup fields. Set Program as the lookup index and keep the student index.
df = students()
lookup = departments().set_index("Program")
result = df.join(lookup, on="Program", how="left", validate="many_to_one")
print(result[["Student_ID", "Program", "Department"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 12
assert result["Advisor"].notna().all()
print("Code checks passed. Review the written explanation as well.")

join can use the right table index as the key. Both forms preserve all 12 students here.

**Quick check:** Which side must have unique Program values?

1. The student table
2. The department lookup
3. Both tables

Answer: 2. The lookup side must be unique for many-to-one matching.

**Explain what you learned:** Choose and validate the matching key.

### M17C02. Keep matches or keep every student

An inner merge keeps only keys found in both tables. A left merge keeps every key from the left table. In this complete assessment lookup, a missing Quiz comes from an unmatched student. With other data, a matched assessment could itself have a missing Quiz; use the merge indicator to test for unmatched records.

**Read:** Which table supplies Quiz?

Answer: The assessment table.

**Predict:** Which registered student disappears from the inner merge?

Answer: S012, Lina, because the assessment table has no S012 row.

**Explain:** Does an unmatched quiz mean the student earned zero?

Answer: No. A missing match is unknown and needs investigation.



In [ ]:
# Predict before running: Which registered student disappears from the inner merge?
result = students().merge(assessment(), on="Student_ID", how="inner", validate="one_to_one")
print(result[["Student_ID", "Name", "Quiz"]])
print("Rows:", len(result))

**Your turn:** Change the merge to left and store the full merged table in merged. Select only students whose Quiz is missing into result, keeping Student_ID, Name, and Quiz, then print result.

For the check, keep these result names: `merged`, `result`.

**Hint:** Use how="left" and then Quiz.isna().

**Watch for this:** Filling an unmatched assessment with zero changes its meaning; agree on a policy first.

In [ ]:
# Change this code to complete the task.
# Question: Change the merge to left and store the full merged table in merged. Select only students whose Quiz is missing into result, keeping Student_ID, Name, and Quiz, then print result.
merged = students().merge(assessment(), on="Student_ID", how="left", validate="one_to_one")
result = merged.loc[merged["Quiz"].isna(), ["Student_ID", "Name", "Quiz"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(merged) == 12
assert result["Student_ID"].tolist() == ["S012"]
print("Code checks passed. Review the written explanation as well.")

The left merge retains 12 students; the missing-match report contains only S012.

**Quick check:** Which merge preserves every registered student?

1. inner
2. left, with students on the left
3. right, with assessments on the right

Answer: 2. A left merge preserves the left table keys even when there is no right-side match.

**Explain what you learned:** Merge type determines which records remain.

### M17C03. Audit both unmatched sides

An outer merge keeps keys from either table. indicator=True adds a _merge column showing left_only, right_only, or both. This is a simple way to find records that need review.

**Read:** What does right_only mean here?

Answer: The key exists in assessments but not in the student table.

**Predict:** Which IDs will the audit show?

Answer: S012 is left_only; S013 is right_only.

**Explain:** Why inspect both unmatched sides?

Answer: One side can reveal missing assessments; the other can reveal assessments for an unregistered or mistyped ID.



In [ ]:
# Predict before running: Which IDs will the audit show?
audit = students().merge(assessment(), on="Student_ID", how="outer", indicator=True, validate="one_to_one")
result = audit.loc[audit["_merge"].ne("both"), ["Student_ID", "_merge"]]
print(result)
print("All merged rows:", len(audit))

**Your turn:** Use a right merge instead. Keep indicator=True and report the row that has no registered student name.

For the check, keep these result names: `audit`, `result`.

**Hint:** A right merge preserves the assessment keys; inspect Name.isna().

**Watch for this:** Pandas may match missing keys to other missing keys; audit or reject null keys before important merges.

In [ ]:
# Change this code to complete the task.
# Question: Use a right merge instead. Keep indicator=True and report the row that has no registered student name.
audit = students().merge(assessment(), on="Student_ID", how="right", indicator=True, validate="one_to_one")
result = audit.loc[audit["Name"].isna(), ["Student_ID", "_merge"]]
print(result)
print("All merged rows:", len(audit))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(audit) == 12
assert result["Student_ID"].tolist() == ["S013"]
print("Code checks passed. Review the written explanation as well.")

The right merge keeps 12 assessment records. S013 remains even though no student profile matches.

**Quick check:** How many distinct keys does the outer merge contain?

1. 11
2. 12
3. 13

Answer: 3. The union contains S001 through S013: 11 shared keys plus 2 unmatched keys.

**Explain what you learned:** Use indicator to make unmatched records visible.

### M17C04. Stop duplicate-key expansion

If a lookup repeats a key, one input row can produce several output rows. The code deliberately adds a duplicate AI lookup. Catching MergeError lets us demonstrate the problem without stopping the lesson.

**Read:** Why is try/except used?

Answer: To catch the expected validation error and continue the lesson.

**Predict:** How many rows appear without validation?

Answer: 18: each of the 6 AI students matches twice, and the 6 IT students match once.

**Explain:** Should every duplicate key be dropped automatically?

Answer: No. Repeated keys may describe different facts. This example repeats an identical lookup row, so removing that exact duplicate is justified.



In [ ]:
# Predict before running: How many rows appear without validation?
bad_lookup = pd.concat([departments(), departments().iloc[[0]]], ignore_index=True)
blocked = False
try:
    students().merge(bad_lookup, on="Program", validate="many_to_one")
except pd.errors.MergeError:
    blocked = True
    print("Blocked: Program is not unique in the lookup.")
result = students().merge(bad_lookup, on="Program")
print("Rows without validation:", len(result))

**Your turn:** Remove only exact duplicate lookup rows, confirm Program is unique, and perform the validated merge.

For the check, keep these result names: `result`.

**Hint:** drop_duplicates() removes exact repeats; is_unique checks the key.

**Watch for this:** validate checks the relationship; it does not decide how to repair conflicting records.

In [ ]:
# Change this code to complete the task.
# Question: Remove only exact duplicate lookup rows, confirm Program is unique, and perform the validated merge.
bad_lookup = pd.concat([departments(), departments().iloc[[0]]], ignore_index=True)
lookup = bad_lookup.drop_duplicates()
assert lookup["Program"].is_unique
result = students().merge(lookup, on="Program", validate="many_to_one")
print("Validated rows:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 12
assert result["Student_ID"].is_unique
print("Code checks passed. Review the written explanation as well.")

The duplicated lookup was identical, so removing the exact repeat restores 12 rows.

**Quick check:** What can repeated keys on both sides cause?

1. A multiplication of matching rows
2. Automatic averaging
3. Automatic removal of duplicates

Answer: 1. Each matching pair produces a row, so duplicate keys can expand the result.

**Explain what you learned:** Check key uniqueness before trusting merged row counts.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates